# Big Data Analytics & Predictive Intelligence

Customer churn prediction pipeline: data loading, RFM feature engineering, model training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

df = pd.read_csv('../dataset/customer_transactions.csv', parse_dates=['transaction_date'])
df.head()

## Feature Engineering — RFM Analysis

In [ ]:
snapshot_date = df['transaction_date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_id').agg(
    recency=('transaction_date', lambda x: (snapshot_date - x.max()).days),
    frequency=('transaction_id', 'count'),
    monetary=('amount', 'sum'),
    avg_order_value=('amount', 'mean'),
    unique_categories=('category', 'nunique'),
    age=('age', 'first'),
    city=('city', 'first'),
    gender=('gender', 'first'),
).reset_index()

rfm['churn'] = (rfm['recency'] > 180).astype(int)
rfm['churn'].value_counts()

## Model Training — Random Forest Churn Classifier

In [ ]:
le_city = LabelEncoder()
le_gender = LabelEncoder()
rfm['city_enc'] = le_city.fit_transform(rfm['city'])
rfm['gender_enc'] = le_gender.fit_transform(rfm['gender'])

features = ['frequency', 'monetary', 'avg_order_value', 'unique_categories', 'age', 'city_enc', 'gender_enc']
X = rfm[features]
y = rfm['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

## Evaluation

In [ ]:
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
importances

## Key Insight

Monetary value and purchase frequency are the strongest predictors of churn — customers with low total spend and infrequent purchases are most at risk. See `../report/Big_Data_Analytics_Report.pdf` for full business recommendations and `../dashboard/dashboard_full.png` for the executive dashboard.